# Source model + AdaBN + DANN for one held-out hospital, in one go

**Before starting:** Runtime → Change runtime type → **T4 GPU**, then **Runtime → Run all**. Authorize Google Drive when asked (first cell), then nothing else to do: about 2.5 h for HH.

What it runs, for `TEST_SITE` (cell 2), on t1-linear images with contrast augmentation and the learning-rate scheduler, seed 42:
1. training of the source model (the two other hospitals), then evaluation of its **final** model, with and without AdaBN;
2. DANN fine-tuning from that model: control (λ = 0), DANN λ = 0.3, IW-DANN λ = 0.3 (re-weighting by age);
3. a summary table.

Everything is saved on Drive (`test-clinicadl/outputs`). If the session disconnects, run all again: finished steps are skipped.
Methods and choices: `scripts/train_dann.py` and `docs/journal.md` in the repository.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import torch
assert torch.cuda.is_available(), 'No GPU: Runtime > Change runtime type > T4 GPU, then Run all again'

In [ ]:
TEST_SITE = 'HH'     # held-out hospital: 'HH', 'Guys' or 'IOP'
SEED = 42
CONFIGS = [(0, False), (0.3, False), (0.3, True)]   # DANN configurations: (lambda_max, iw)

In [ ]:
# scripts from GitHub, outputs on Drive
import os
DRIVE = '/content/drive/MyDrive/test-clinicadl'
WORK = '/content/test-clinicadl'
REPO = 'https://github.com/Ayoub-TAREK/ixi-brain-age.git'
if os.path.exists(WORK) and not os.path.exists(f'{WORK}/.git'):
    !rm -rf {WORK}
if not os.path.exists(WORK):
    !git clone -q {REPO} {WORK}
else:
    !git -C {WORK} pull -q
os.makedirs(f'{DRIVE}/outputs', exist_ok=True)
if not os.path.exists(f'{WORK}/outputs'):
    os.symlink(f'{DRIVE}/outputs', f'{WORK}/outputs')
!git -C {WORK} log --oneline -1

In [ ]:
# t1-linear images (2 mm): from the .tar on Drive, or downloaded with its share link (~1 min)
import glob
T1L_FILE_ID = '1mdWBPxt_DrgNbCdlKAC_7HxAU7twkl3D'
T1L_TAR = f'{DRIVE}/caps_t1linear_2mm.tar'
T1L_DIR = f'{WORK}/data/IXI/caps_t1linear_2mm'
if not glob.glob(f'{T1L_DIR}/subjects/*/*/t1_linear/*_T1w.nii.gz'):
    os.makedirs(f'{WORK}/data/IXI', exist_ok=True)
    if os.path.exists(T1L_TAR):
        !tar -xf {T1L_TAR} -C {WORK}/data/IXI
    else:
        !pip -q install gdown
        !gdown -q {T1L_FILE_ID} -O /content/caps_t1linear_2mm.tar
        !tar -xf /content/caps_t1linear_2mm.tar -C {WORK}/data/IXI && rm /content/caps_t1linear_2mm.tar
n = len(glob.glob(f'{T1L_DIR}/subjects/*/*/t1_linear/*_res-2x2x2_T1w.nii.gz'))
assert n == 563, f'{n} images found instead of 563'
print(n, 't1-linear images')

In [ ]:
!pip install -q clinicadl==2.0.0 xlrd
!python -c "import clinicadl, torch; print('ClinicaDL', clinicadl.__version__, '| CUDA', torch.cuda.is_available())" 

In [ ]:
# 1. source model (~30-50 min), then its final model with and without AdaBN
import shutil, time
from IPython.display import Image, display
%cd {WORK}
T0 = time.time()
SRC = f'--data t1linear --test-site {TEST_SITE} --augment --scheduler plateau --seed {SEED}'
src_dir = f'{WORK}/outputs/site_shift_test-{TEST_SITE}_t1linear_aug_plateau' + ('' if SEED == 42 else f'_seed{SEED}')

if os.path.exists(f'{src_dir}/predictions_adabn_final.tsv'):
    print('source model already done, skipped:', os.path.basename(src_dir))
else:
    for name in ['maps', 'split', 'train_sites.tsv', 'out_of_site.tsv']:   # unfinished earlier attempt
        path = os.path.join(src_dir, name)
        if os.path.isdir(path):
            shutil.rmtree(path)
        elif os.path.exists(path):
            os.remove(path)
    !python scripts/train_ixi_site_shift.py {SRC} --epochs 50 --batch-size 8 2>&1 | grep -E "seed :|scheduler|Epoch [0-9]+ completed|stopped|Error|Traceback"
    assert os.path.exists(f'{src_dir}/maps/training/split-0/models/final/model.pt'), 'training failed: see above'
    !python scripts/predict_ages.py {SRC} --checkpoint final 2>&1 | grep -v -i warning | tail -n 12
    !python scripts/adabn.py {SRC} --checkpoint final 2>&1 | grep -v -i warning | tail -n 7
    assert os.path.exists(f'{src_dir}/predictions_adabn_final.tsv'), 'evaluation failed: see above'
display(Image(f'{src_dir}/validation_curve.png'))
print(f'{(time.time() - T0) / 60:.0f} min')

In [ ]:
# 2. DANN configurations (~30-40 min each), starting from the final source model
for lam, iw in CONFIGS:
    run = f"{src_dir}_dann{lam:g}{'_iw' if iw else ''}"
    if os.path.exists(f'{run}/predictions_final.tsv'):
        print('already done, skipped:', os.path.basename(run))
        continue
    print(f'\n===== {os.path.basename(run)} =====')
    iw_flag = '--iw' if iw else ''
    !python scripts/train_dann.py {SRC} --lambda-max {lam} {iw_flag} 2>&1 | grep -v -i -E "warning|GradScaler|cpu_feature_guard|rebuild TensorFlow"
    assert os.path.exists(f'{run}/maps/training/split-0/models/final/model.pt'), 'DANN failed: see above'
    !python scripts/predict_ages.py --run-dir {run} --data t1linear --checkpoint final 2>&1 | grep -v -i warning | tail -n 7
    display(Image(f'{run}/dann_curves.png'))
print(f'total: {(time.time() - T0) / 60:.0f} min')

In [ ]:
# 3. summary: out-of-site results of every method (final models), and IW-DANN weights
import pandas as pd

def scores(tsv, group, name):
    df = pd.read_csv(tsv, sep='\t')
    d, s = df[df.group == group], df[df.group == 'insite']
    return {'method': name, 'MAE insite': s.error.abs().mean(), 'MAE outofsite': d.error.abs().mean(),
            'bias outofsite': d.error.mean(), 'r outofsite': d.AgeMRI.corr(d.predicted)}

rows = [scores(f'{src_dir}/predictions_adabn_final.tsv', 'outofsite', 'source model'),
        scores(f'{src_dir}/predictions_adabn_final.tsv', 'outofsite_adabn', 'AdaBN')]
for lam, iw in CONFIGS:
    run = f"{src_dir}_dann{lam:g}{'_iw' if iw else ''}"
    if os.path.exists(f'{run}/predictions_final.tsv'):
        name = 'control (lambda 0)' if lam == 0 else f"{'IW-DANN' if iw else 'DANN'} lambda {lam:g}"
        row = scores(f'{run}/predictions_final.tsv', 'outofsite', name)
        log = pd.read_csv(f'{run}/dann_log.tsv', sep='\t')
        row['domain acc (end)'] = log.domain_acc.iloc[-1]
        row['mean predicted age (end)'] = log.target_pred_mean.iloc[-1]
        rows.append(row)
ages = pd.read_csv(f'{T1L_DIR}/participants_sessions_age.tsv', sep='\t')
print(f"{TEST_SITE}: true mean age {ages[ages.site == TEST_SITE].AgeMRI.mean():.1f} | "
      f"training hospitals: {ages[ages.site != TEST_SITE].AgeMRI.mean():.1f}  (for interpretation only)")
display(pd.DataFrame(rows).round(2))
iw_log = f'{src_dir}_dann0.3_iw/dann_log.tsv'
if os.path.exists(iw_log):
    print('IW-DANN weights per age bin (w0 = youngest), per epoch:')
    display(pd.read_csv(iw_log, sep='\t')[['epoch', 'target_pred_mean', 'w0', 'w1', 'w2', 'w3', 'w4']].round(2))